In [ ]:
import sqlite3

# Connect to the db
conn = sqlite3.connect('travel_planner.db')

# Create the cursor
cursor = conn.cursor()

# I'm updating activities. I'll add 10 activities
new_activities = ("Budget backpacking", "Beach Vacation", "Cruise Vacation", "Road Tripping", "Camping Trip", "Ski Holiday", "Sightseeing Tour", "Resort Stay", "Food Touring", "Heritage Travel")

# Now we execute the change using the cursor
cursor.execute("INSERT INTO Activities (ActivityName) VALUES (?)", new_activities)

new_spotlight_description = ("nothing_x", "nothing_x2", "nothing_x3", "nothing_x4", "nothing_x5", "nothing_x6", "nothing_x7", "nothing_x8", "nothing_x9", "nothing_x10")

cursor.execute("INSERT INTO Destinations_Activities (Spotlight_Description) VALUES (?)", new_spotlight_description)

new_travel_vibe = ("Frugal Adventure", "Sun & Sand", "Nautical Luxury", "Open Road", "Rustic Wilderness", "Alpine Thrills", "Urban Explorer", "Pure Relaxation", "Gastronomic Journey", "Cultural Roots")
cursor.execute("INSERT INTO Travel_Vibes (VibeName) VALUES (?)", new_travel_vibe)

# Let's add cities:

cities_to_add = [
("Tokyo", "Japan", "The capital of Japan, a beautiful place to visit anytime."),
("Bucharest", "Romania", "The capital of Romania, it's alright."),
("London", "England", "The capital of England, it's a nice place for pubcrawling."),
("Paris", "France", "The capital of France, iconic for food, art, and cafes."),
("Rome", "Italy", "The capital of Italy, packed with ancient history and incredible pasta."),
("New York", "USA", "The Big Apple, famous for its non-stop energy and Broadway shows."),
("Barcelona", "Spain", "A vibrant coastal city known for stunning architecture and tapas."),
("Bangkok", "Thailand", "A bustling tropical hub famous for street food and ornate temples."),
("Sydney", "Australia", "A gorgeous harbor city with amazing beaches and a laid-back vibe."),
("Cairo", "Egypt", "A historic desert metropolis home to the ancient pyramids.")
]

# Now bcs we have multiple arrays, we need to use a loop to add everything
cursor.executemany("""
    INSERT INTO Destinations (CityName, Country, Description)
    VALUES (?, ?, ?)
""", cities_to_add)

conn.commit()
print("Data succesfully saved!")

# Now we close it
conn.close()

In [ ]:
import sqlite3

# Connect to the db
conn = sqlite3.connect('travel_planner.db')
cursor = conn.cursor()

# 1. ADDING ACTIVITIES
# Formatted as a LIST of tuples so we can use executemany
new_activities = [
    ("Budget backpacking",), ("Beach Vacation",), ("Cruise Vacation",), 
    ("Road Tripping",), ("Camping Trip",), ("Ski Holiday",), 
    ("Sightseeing Tour",), ("Resort Stay",), ("Food Touring",), 
    ("Heritage Travel",)
]

cursor.executemany("INSERT INTO Activities (ActivityName) VALUES (?)", new_activities)


# 2. ADDING TRAVEL VIBES
# Formatted as a LIST of tuples
new_travel_vibes = [
    ("Frugal Adventure",), ("Sun & Sand",), ("Nautical Luxury",), 
    ("Open Road",), ("Rustic Wilderness",), ("Alpine Thrills",), 
    ("Urban Explorer",), ("Pure Relaxation",), ("Gastronomic Journey",), 
    ("Cultural Roots",)
]

cursor.executemany("INSERT INTO Travel_Vibes (VibeName) VALUES (?)", new_travel_vibes)


# 3. ADDING CITIES
# Your formatting here was already absolutely perfect!
cities_to_add = [
    ("Tokyo", "Japan", "The capital of Japan, a beautiful place to visit anytime."),
    ("Bucharest", "Romania", "The capital of Romania, it's alright."),
    ("London", "England", "The capital of England, it's a nice place for pubcrawling."),
    ("Paris", "France", "The capital of France, iconic for food, art, and cafes."),
    ("Rome", "Italy", "The capital of Italy, packed with ancient history and incredible pasta."),
    ("New York", "USA", "The Big Apple, famous for its non-stop energy and Broadway shows."),
    ("Barcelona", "Spain", "A vibrant coastal city known for stunning architecture and tapas."),
    ("Bangkok", "Thailand", "A bustling tropical hub famous for street food and ornate temples."),
    ("Sydney", "Australia", "A gorgeous harbor city with amazing beaches and a laid-back vibe."),
    ("Cairo", "Egypt", "A historic desert metropolis home to the ancient pyramids.")
]

cursor.executemany("""
    INSERT INTO Destinations (CityName, Country, Description)
    VALUES (?, ?, ?)
""", cities_to_add)


# Save and close!
conn.commit()
print("Data successfully saved!")
conn.close()

In [ ]:
import sqlite3

# Connect to the db
conn = sqlite3.connect('travel_planner.db')
cursor = conn.cursor()

# We are linking DestinationID to VibeID
# Example: (1, 7) means Tokyo (1) gets the "Urban Explorer" (7) vibe
# Example: (1, 9) means Tokyo (1) also gets "Gastronomic Journey" (9)

vibe_links = [
    (1, 7), (1, 9), # Tokyo
    (2, 1), (2, 7), # Bucharest 
    (3, 7), (3, 10),# London
    (4, 9), (4, 10),# Paris
    (5, 9), (5, 10),# Rome
    (6, 7),         # New York
    (7, 2), (7, 9), # Barcelona
    (8, 1), (8, 9), # Bangkok
    (9, 2), (9, 8), # Sydney
    (10, 1), (10, 10) # Cairo
]

# Insert the pairs into the purely relational link table
cursor.executemany("""
    INSERT INTO Destination_Vibes (DestinationID, VibeID) 
    VALUES (?, ?)
""", vibe_links)

# Let's also populate Destinations_Activities while we are here!
# This requires 3 pieces of data: DestinationID, ActivityID, and Spotlight_Description
activity_links = [
    (1, 9, "Eat your way through the Tsukiji Outer Market."), # Tokyo -> Food Touring
    (4, 10, "Spend days getting lost in the Louvre and Musée d'Orsay."), # Paris -> Heritage Travel
    (8, 1, "Backpack through Khao San Road for the ultimate budget trip.") # Bangkok -> Budget Backpacking
]

cursor.executemany("""
    INSERT INTO Destinations_Activities (DestinationID, ActivityID, Spotlight_Description) 
    VALUES (?, ?, ?)
""", activity_links)

# Save and close!
conn.commit()
print("Link tables successfully populated!")
conn.close()

In [ ]:
# Let's visualise it using panda
import sqlite3
import pandas as pd

conn = sqlite3.connect('travel_planner.db')

# The Query: Let's find all the vibes associated with Tokyo
# 1. We SELECT the columns we actually want to read
# 2. We FROM the main table
# 3. We JOIN the tables together by matching the Primary Keys to the Foreign Keys
# 4. We use WHERE to filter down to just City ID 1 (Tokyo)

query = """
    SELECT 
        Destinations.CityName, 
        Travel_Vibes.VibeName
    FROM Destinations
    JOIN Destination_Vibes ON Destinations.DestinationID = Destination_Vibes.DestinationID
    JOIN Travel_Vibes ON Destination_Vibes.VibeID = Travel_Vibes.VibeID
    WHERE Destinations.DestinationID = 4;
"""

# Ask pandas to run it and display it!
df_results = pd.read_sql_query(query, conn)
conn.close()

display(df_results)



In [ ]:
import sqlite3

# Connect to the db
conn = sqlite3.connect('travel_planner.db')
cursor = conn.cursor()

# 1. ADDING ACTIVITIES
new_activities = [
    ("Budget backpacking",), ("Beach Vacation",), ("Cruise Vacation",), 
    ("Road Tripping",), ("Camping Trip",), ("Ski Holiday",), 
    ("Sightseeing Tour",), ("Resort Stay",), ("Food Touring",), 
    ("Heritage Travel",)
]
cursor.executemany("INSERT INTO Activities (ActivityName) VALUES (?)", new_activities)

# 2. ADDING TRAVEL VIBES
new_travel_vibes = [
    ("Frugal Adventure",), ("Sun & Sand",), ("Nautical Luxury",), 
    ("Open Road",), ("Rustic Wilderness",), ("Alpine Thrills",), 
    ("Urban Explorer",), ("Pure Relaxation",), ("Gastronomic Journey",), 
    ("Cultural Roots",)
]
cursor.executemany("INSERT INTO Travel_Vibes (VibeName) VALUES (?)", new_travel_vibes)

# 3. ADDING CITIES
cities_to_add = [
    ("Tokyo", "Japan", "The capital of Japan, a beautiful place to visit anytime."),
    ("Bucharest", "Romania", "The capital of Romania, it's alright."),
    ("London", "England", "The capital of England, it's a nice place for pubcrawling."),
    ("Paris", "France", "The capital of France, iconic for food, art, and cafes."),
    ("Rome", "Italy", "The capital of Italy, packed with ancient history and incredible pasta."),
    ("New York", "USA", "The Big Apple, famous for its non-stop energy and Broadway shows."),
    ("Barcelona", "Spain", "A vibrant coastal city known for stunning architecture and tapas."),
    ("Bangkok", "Thailand", "A bustling tropical hub famous for street food and ornate temples."),
    ("Sydney", "Australia", "A gorgeous harbor city with amazing beaches and a laid-back vibe."),
    ("Cairo", "Egypt", "A historic desert metropolis home to the ancient pyramids.")
]
cursor.executemany("""
    INSERT INTO Destinations (CityName, Country, Description)
    VALUES (?, ?, ?)
""", cities_to_add)

# --- NEW DATA FOR THE RESTRUCTURED DATABASE ---

# 4. ADDING COST PROFILES (One-to-One: 10 destinations = 10 cost profiles)
# Format: (DestinationID, BudgetLevel)
cost_profiles_to_add = [
    (1, "Expensive"), (2, "Cheap"),      (3, "Expensive"), 
    (4, "Expensive"), (5, "Medium"),     (6, "Expensive"), 
    (7, "Medium"),    (8, "Cheap"),      (9, "Expensive"), 
    (10, "Cheap")
]
cursor.executemany("""
    INSERT INTO Cost_Profiles (DestinationID, BudgetLevel)
    VALUES (?, ?)
""", cost_profiles_to_add)


# 5. ADDING WEATHER RECORDS (One-to-Many: adding July data for all 10 cities)
# Format: (DestinationID, Month, AvgTempC, RainfallMM)
weather_to_add = [
    (1, 7, 26.5, 150.0), (2, 7, 23.0, 60.0),  (3, 7, 19.0, 45.0),
    (4, 7, 21.0, 55.0),  (5, 7, 25.5, 20.0),  (6, 7, 25.0, 100.0),
    (7, 7, 24.5, 25.0),  (8, 7, 29.5, 170.0), (9, 7, 13.0, 95.0), 
    (10, 7, 28.0, 0.0)
]
cursor.executemany("""
    INSERT INTO Weather_Monthly (DestinationID, Month, AvgTempC, RainfallMM)
    VALUES (?, ?, ?, ?)
""", weather_to_add)


# 6. LINKING DESTINATIONS TO ACTIVITIES (10 links)
# Format: (DestinationID, ActivityID, Spotlight_Description)
dest_activities_to_add = [
    (1, 9, "Eat your way through the Tsukiji Outer Market."), 
    (2, 1, "Explore old town streets on a tight budget."),
    (3, 7, "See Big Ben and the Tower of London."),
    (4, 10, "Spend days getting lost in the Louvre and Musée d'Orsay."),
    (5, 7, "Marvel at the Colosseum and the Pantheon."),
    (6, 7, "Catch a Broadway show in Times Square."),
    (7, 2, "Relax on Barceloneta Beach."),
    (8, 1, "Backpack through Khao San Road for the ultimate budget trip."),
    (9, 2, "Surf at Bondi Beach."),
    (10, 10, "Ride a camel near the Pyramids of Giza.")
]
cursor.executemany("""
    INSERT INTO Destinations_Activities (DestinationID, ActivityID, Spotlight_Description)
    VALUES (?, ?, ?)
""", dest_activities_to_add)


# 7. LINKING DESTINATIONS TO VIBES (10 links)
# Format: (DestinationID, VibeID)
dest_vibes_to_add = [
    (1, 7),  # Tokyo -> Urban Explorer
    (2, 1),  # Bucharest -> Frugal Adventure
    (3, 7),  # London -> Urban Explorer
    (4, 9),  # Paris -> Gastronomic Journey
    (5, 10), # Rome -> Cultural Roots
    (6, 7),  # New York -> Urban Explorer
    (7, 2),  # Barcelona -> Sun & Sand
    (8, 1),  # Bangkok -> Frugal Adventure
    (9, 2),  # Sydney -> Sun & Sand
    (10, 10) # Cairo -> Cultural Roots
]
cursor.executemany("""
    INSERT INTO Destination_Vibes (DestinationID, VibeID)
    VALUES (?, ?)
""", dest_vibes_to_add)

# Save and close!
conn.commit()
print("Data successfully saved!")
conn.close()

In [3]:
%pip install requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [requests]
Note: you may need to restart the kernel to use updated packages.


In [5]:
import sqlite3
import requests
import time
from datetime import datetime

# Connect to the db
conn = sqlite3.connect('travel_planner.db')
cursor = conn.cursor()

# 1. ADDING ACTIVITIES
new_activities = [
    ("Budget backpacking",), ("Beach Vacation",), ("Cruise Vacation",), 
    ("Road Tripping",), ("Camping Trip",), ("Ski Holiday",), 
    ("Sightseeing Tour",), ("Resort Stay",), ("Food Touring",), 
    ("Heritage Travel",)
]
cursor.executemany("INSERT INTO Activities (ActivityName) VALUES (?)", new_activities)

# 2. ADDING TRAVEL VIBES
new_travel_vibes = [
    ("Frugal Adventure",), ("Sun & Sand",), ("Nautical Luxury",), 
    ("Open Road",), ("Rustic Wilderness",), ("Alpine Thrills",), 
    ("Urban Explorer",), ("Pure Relaxation",), ("Gastronomic Journey",), 
    ("Cultural Roots",)
]
cursor.executemany("INSERT INTO Travel_Vibes (VibeName) VALUES (?)", new_travel_vibes)

# 3. ADDING CITIES
cities_to_add = [
    ("Tokyo", "Japan", "The capital of Japan, a beautiful place to visit anytime."),
    ("Bucharest", "Romania", "The capital of Romania, it's alright."),
    ("London", "England", "The capital of England, it's a nice place for pubcrawling."),
    ("Paris", "France", "The capital of France, iconic for food, art, and cafes."),
    ("Rome", "Italy", "The capital of Italy, packed with ancient history and incredible pasta."),
    ("New York", "USA", "The Big Apple, famous for its non-stop energy and Broadway shows."),
    ("Barcelona", "Spain", "A vibrant coastal city known for stunning architecture and tapas."),
    ("Bangkok", "Thailand", "A bustling tropical hub famous for street food and ornate temples."),
    ("Sydney", "Australia", "A gorgeous harbor city with amazing beaches and a laid-back vibe."),
    ("Cairo", "Egypt", "A historic desert metropolis home to the ancient pyramids.")
]
cursor.executemany("""
    INSERT INTO Destinations (CityName, Country, Description)
    VALUES (?, ?, ?)
""", cities_to_add)

# --- NEW DATA FOR THE RESTRUCTURED DATABASE ---

# 4. ADDING COST PROFILES (One-to-One: 10 destinations = 10 cost profiles)
cost_profiles_to_add = [
    (1, "Expensive"), (2, "Cheap"),      (3, "Expensive"), 
    (4, "Expensive"), (5, "Medium"),     (6, "Expensive"), 
    (7, "Medium"),    (8, "Cheap"),      (9, "Expensive"), 
    (10, "Cheap")
]
cursor.executemany("""
    INSERT INTO Cost_Profiles (DestinationID, BudgetLevel)
    VALUES (?, ?)
""", cost_profiles_to_add)


# 5. FETCHING WEATHER RECORDS VIA OPEN-METEO (100% Free, no API key needed)
print("Fetching 12 months of historical weather data. This will take a few seconds...")

cursor.execute("SELECT DestinationID, CityName FROM Destinations")
saved_cities = cursor.fetchall()

for city in saved_cities:
    dest_id = city[0]
    city_name = city[1]
    
    try:
        # Step A: Get coordinates (Latitude/Longitude) for the city
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city_name}&count=1&format=json"
        geo_data = requests.get(geo_url).json()
        
        if "results" not in geo_data or not geo_data["results"]:
            print(f"Could not find coordinates for {city_name}")
            continue
            
        lat = geo_data["results"][0]["latitude"]
        lon = geo_data["results"][0]["longitude"]
        
        # Step B: Get historical daily data for a full year (2023)
        archive_url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date=2023-01-01&end_date=2023-12-31&daily=temperature_2m_mean,precipitation_sum"
        weather_data = requests.get(archive_url).json()
        
        dates = weather_data["daily"]["time"]
        temps = weather_data["daily"]["temperature_2m_mean"]
        rains = weather_data["daily"]["precipitation_sum"]
        
        # Step C: Group daily data into monthly arrays
        monthly_temps = {m: [] for m in range(1, 13)}
        monthly_rains = {m: [] for m in range(1, 13)}
        
        for d, t, r in zip(dates, temps, rains):
            if t is not None and r is not None:
                month = datetime.strptime(d, "%Y-%m-%d").month
                monthly_temps[month].append(t)
                monthly_rains[month].append(r)
                
        # Step D: Average the data out and save to database
        for month in range(1, 13):
            avg_temp = sum(monthly_temps[month]) / len(monthly_temps[month])
            total_rain = sum(monthly_rains[month])
            
            cursor.execute("""
                INSERT INTO Weather_Monthly (DestinationID, Month, AvgTempC, RainfallMM)
                VALUES (?, ?, ?, ?)
            """, (dest_id, month, round(avg_temp, 1), round(total_rain, 1)))
            
        print(f"Saved 12 months of weather data for {city_name}!")
        
    except Exception as e:
        print(f"Failed to get data for {city_name}. Error: {e}")
        
    # Be polite to the free API so we don't get blocked
    time.sleep(0.5) 


# 6. LINKING DESTINATIONS TO ACTIVITIES (10 links)
dest_activities_to_add = [
    (1, 9, "Eat your way through the Tsukiji Outer Market."), 
    (2, 1, "Explore old town streets on a tight budget."),
    (3, 7, "See Big Ben and the Tower of London."),
    (4, 10, "Spend days getting lost in the Louvre and Musée d'Orsay."),
    (5, 7, "Marvel at the Colosseum and the Pantheon."),
    (6, 7, "Catch a Broadway show in Times Square."),
    (7, 2, "Relax on Barceloneta Beach."),
    (8, 1, "Backpack through Khao San Road for the ultimate budget trip."),
    (9, 2, "Surf at Bondi Beach."),
    (10, 10, "Ride a camel near the Pyramids of Giza.")
]
cursor.executemany("""
    INSERT INTO Destinations_Activities (DestinationID, ActivityID, Spotlight_Description)
    VALUES (?, ?, ?)
""", dest_activities_to_add)


# 7. LINKING DESTINATIONS TO VIBES (10 links)
dest_vibes_to_add = [
    (1, 7),  # Tokyo -> Urban Explorer
    (2, 1),  # Bucharest -> Frugal Adventure
    (3, 7),  # London -> Urban Explorer
    (4, 9),  # Paris -> Gastronomic Journey
    (5, 10), # Rome -> Cultural Roots
    (6, 7),  # New York -> Urban Explorer
    (7, 2),  # Barcelona -> Sun & Sand
    (8, 1),  # Bangkok -> Frugal Adventure
    (9, 2),  # Sydney -> Sun & Sand
    (10, 10) # Cairo -> Cultural Roots
]
cursor.executemany("""
    INSERT INTO Destination_Vibes (DestinationID, VibeID)
    VALUES (?, ?)
""", dest_vibes_to_add)

# Save and close!
conn.commit()
print("All data successfully saved!")
conn.close()

Fetching 12 months of historical weather data. This will take a few seconds...
Saved 12 months of weather data for Tokyo!
Saved 12 months of weather data for Bucharest!
Saved 12 months of weather data for London!
Saved 12 months of weather data for Paris!
Saved 12 months of weather data for Rome!
Saved 12 months of weather data for New York!
Saved 12 months of weather data for Barcelona!
Saved 12 months of weather data for Bangkok!
Saved 12 months of weather data for Sydney!
Saved 12 months of weather data for Cairo!
All data successfully saved!
